# Provincia de Santiago — App interactiva de morfología Liu

Mapa web con **diseño depurado** para que la grilla y las calles se lean sin confusión:

- **Parches Liu como coropleta limpia**: relleno por clase con bordes blancos sutiles
  (sin líneas discontinuas), resaltado al pasar el cursor, popup con probabilidades.
- **Red vial monocroma por jerarquía**: solo vías estructurantes (Expresa/Troncal/Colectora),
  en gris oscuro con grosor según jerarquía — no compiten con los colores de la coropleta.
- **Límites de comunas** con nombre al pasar el cursor, para orientarse.
- **Mediciones acústicas reales** (AUMILAB, enero 2026) coloreadas por LA_eq.
- Render en *canvas* para mover miles de polígonos con fluidez.

Resultado: `app_interactiva/mapa_santiago_liu.html`.


In [13]:
# ============================================================
# 1. Cargar datos
# ============================================================

# from scan_completo.paso1_clasificacion_morfologia import DATOS
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd
import folium
from folium import FeatureGroup, LayerControl
from shapely.ops import unary_union, linemerge

NOMBRE_CIUDAD = "santiago"  # Mismo identificador usado en la descarga y en el paso 1
RADIO_SCAN_M  = 150.0        # 🎯 Radio de análisis del sensor en metros (ej. 150, 300, etc.)

# DATOS = Path("02_datos")
DATOS = Path("00_descarga")
MODELO_01 = Path("01_clas")
MODELO_01_IN = Path("01_clas/00_in")
APP = Path("03_app")
APP.mkdir(exist_ok=True)
DATOS_02 = Path("02_ana")

with open(MODELO_01_IN / "clases_liu.json", encoding="utf-8") as f:
    meta_liu = json.load(f)
CLASES = {v["nombre"]: v for v in meta_liu["clases"].values()}

gdf_parches = gpd.read_file(MODELO_01 / f"parches_liu_{NOMBRE_CIUDAD.lower()}.geojson")
prov = gpd.read_file(DATOS / f"limite_{NOMBRE_CIUDAD.lower()}.geojson")
comunas = gpd.read_file(DATOS / f"comunas_{NOMBRE_CIUDAD.lower()}.geojson")
# quedarse solo con comunas de la provincia (la consulta OSM trae de todo Chile)
poligono_prov = prov.geometry.iloc[0]
comunas = comunas[comunas.representative_point().within(poligono_prov.buffer(0.005))]
if "name" in comunas.columns:
    comunas = comunas.dropna(subset=["name"])
    # excluir la relación de la provincia misma y duplicados
    comunas = comunas[~comunas["name"].str.contains("Provincia", case=False)]
    comunas = comunas.drop_duplicates(subset=["name"])
edges = gpd.read_parquet(DATOS / f"red_drive_{NOMBRE_CIUDAD.lower()}.parquet")

with open(MODELO_01 / f"mediciones_aumilab_{NOMBRE_CIUDAD.lower()}.json", encoding="utf-8") as f:
    mediciones = json.load(f)

path_ml = Path(DATOS_02 / f"ml_dataset_{NOMBRE_CIUDAD.lower()}.json")
ml_dict = {}
if path_ml.exists():
    with open(path_ml, "r", encoding="utf-8") as f:
        ml_data = json.load(f)
    ml_dict = {str(row["scan_id"]): row for row in ml_data}

print("Parches:", len(gdf_parches), "| Comunas:", len(comunas),
      "| Aristas:", len(edges), "| Mediciones:", len(mediciones))


Parches: 10811 | Comunas: 33 | Aristas: 238238 | Mediciones: 34


In [14]:
# ============================================================
# 2. Red vial estructurante: disolver por jerarquía (pocas features)
# ============================================================

JERARQUIAS = {
    "Expresa":   (["motorway", "motorway_link", "trunk", "trunk_link"], 4.0, 0.80),
    "Troncal":   (["primary", "primary_link"], 2.6, 0.70),
    "Colectora": (["secondary", "secondary_link"], 1.4, 0.55),
}

vias = {}
for nombre, (tags, peso, opacidad) in JERARQUIAS.items():
    sub = edges[edges["highway_osm"].isin(tags)]
    if len(sub) == 0:
        continue
    fusion = linemerge(unary_union(sub.geometry.values))
    g = gpd.GeoSeries([fusion], crs=3857).to_crs(4326).iloc[0]
    vias[nombre] = {"geom": g, "peso": peso, "opacidad": opacidad, "n": len(sub)}
    print(f"{nombre}: {len(sub)} aristas -> 1 geometría fusionada")


Expresa: 1349 aristas -> 1 geometría fusionada
Troncal: 8234 aristas -> 1 geometría fusionada
Colectora: 17004 aristas -> 1 geometría fusionada


In [15]:
# ============================================================
# 3. Construir el mapa (canvas, base clara)
# ============================================================

centro = [prov.geometry.iloc[0].centroid.y, prov.geometry.iloc[0].centroid.x]
m = folium.Map(location=centro, zoom_start=11, tiles=None,
               prefer_canvas=True, control_scale=True)
folium.TileLayer("cartodbpositron", name="Base claro").add_to(m)
folium.TileLayer("openstreetmap", name="OpenStreetMap", show=False).add_to(m)

# ---- Comunas (se agregan primero: quedan DEBAJO de los parches para que
# ---- el hover de los parches tenga prioridad; el nombre de la comuna solo
# ---- aparece donde no hay parches, p. ej. zonas rurales) ----
fg_com = FeatureGroup(name="Comunas", show=True)
folium.GeoJson(
    json.loads(comunas.to_json()),
    style_function=lambda feat: {"fill": False, "color": "#4a4a4a",
                                  "weight": 1.1, "dashArray": "5,5", "opacity": 0.85},
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Comuna"],
                                   style="font-family: system-ui; font-size: 12px;"),
).add_to(fg_com)
fg_com.add_to(m)

# ---- Coropleta de parches Liu ----
def estilo_parche(feat):
    p = feat["properties"]
    info = CLASES.get(p["liu_class"], {})
    return {
        "fillColor": info.get("color", "#888888"),
        "fillOpacity": 0.30 if p["liu_class"] == "Sparse" else 0.58,
        "color": "#ffffff", "weight": 0.6, "opacity": 0.6,
    }

def resaltar(feat):
    return {"fillOpacity": 0.85, "color": "#222222", "weight": 1.6, "opacity": 1.0}

fg_liu = FeatureGroup(name="Morfología", show=True)
gj_parches = folium.GeoJson(
    json.loads(gdf_parches.to_json()),
    style_function=estilo_parche,
    highlight_function=resaltar,
    smooth_factor=0.5,
    tooltip=folium.GeoJsonTooltip(
        fields=["liu_class", "confianza", "n_bloques"],
        aliases=["Clase", "Confianza", "Bloques"],                      # Bloques son las calles que se intersectan de los cuadros alrededor
        style="font-family: system-ui; font-size: 12px;",
    ),
)
gj_parches.add_to(fg_liu)
fg_liu.add_to(m)

# ---- Red vial estructurante (monocroma) ----
fg_vias = FeatureGroup(name="Red vial estructurante", show=True)
for nombre, v in vias.items():
    folium.GeoJson(
        gpd.GeoSeries([v["geom"]]).__geo_interface__,
        style_function=lambda feat, v=v: {
            "color": "#2d2d2d", "weight": v["peso"], "opacity": v["opacidad"]},
        smooth_factor=1.5,
    ).add_to(fg_vias)
fg_vias.add_to(m)

# ---- Mediciones acústicas reales ----
def color_db(db):
    if db >= 70: return "#c62828"
    if db >= 65: return "#ef6c00"
    if db >= 60: return "#f9a825"
    return "#2e7d32"

fg_med = FeatureGroup(name="Mediciones acústicas AUMILAB", show=True)
for r in mediciones:
    scan_id = str(r["id"])
    html_scan = ""
    if scan_id in ml_dict:
        m_row = ml_dict[scan_id]
        html_scan = (
            f"<hr style='margin:8px 0;border-top:1px solid #ccc;'/>"
            f"<div style='font-weight:600;margin-bottom:4px;'>Entorno del Sensor (Radio {m_row.get('scan_radius_mts', {RADIO_SCAN_M})}m)</div>"
            f"<table>"
            f"<tr><td style='padding-right:8px;color:#555'>Total Calles</td><td>{m_row.get('largo_calles_total_m', 0):.1f} m ({m_row.get('cantidad_calles', 0)} calles)</td></tr>"
            f"<tr><td style='padding-right:8px;color:#555'>Secciones (Split)</td><td>{m_row.get('cantidad_tramos_interseccion', 0)} tramos (prom. {m_row.get('largo_promedio_tramo_interseccion_m', 0):.1f} m)</td></tr>"
            f"<tr><td style='padding-right:8px;color:#555'>Densidad Vial</td><td>{m_row.get('densidad_vial_m_por_m2', 0):.5f} m/m²</td></tr>"
            f"</table>"
            f"<div style='font-weight:600;margin:6px 0 2px;'>Morfología Liu en Entorno</div>"
            f"<table>"
        )
        clases_en_scan = ["regular_grid", "irregular_grid", "broken_grid", "warped_parallel", "loops_y_lollipops", "sparse"]
        for c_key in clases_en_scan:
            pct = m_row.get(f"porcentaje_parche_{c_key}", 0)
            area = m_row.get(f"area_parche_{c_key}_m2", 0)
            if pct > 0:
                clase_nom = c_key.replace("_", " ").title().replace("Y", "&")
                html_scan += f"<tr><td style='padding-right:8px;color:#555'>{clase_nom}</td><td><b>{pct:.1f}%</b> ({area:.0f} m²)</td></tr>"
        html_scan += "</table>"

    popup = (f"<div style='font-family:system-ui;font-size:12px;min-width:260px'>"
             f"<h4 style='margin:0 0 4px'>{r['id']} - {r['calle']}</h4>"
             f"<table>"
             f"<tr><td style='padding-right:8px;color:#555'>LA_eq</td><td><b>{r['la_eq']} dB</b></td></tr>"
             f"<tr><td style='padding-right:8px;color:#555'>LA_min / LA_max</td><td>{r['la_min']} / {r['la_max']} dB</td></tr>"
             #f"<tr><td style='padding-right:8px;color:#555'>LA_90 (fondo)</td><td>{r['la_90']} dB</td></tr>"       Esta medición estaba en santiago pero no en valdivia, se quita
             f"<tr><td style='padding-right:8px;color:#555'>Vía OGUC</td><td>{r.get('categoria_oguc', 's/d')}</td></tr>"
             f"<tr><td style='padding-right:8px;color:#555'>Fecha</td><td>{r['fecha']} ({r['periodo']})</td></tr>"
             f"</table>"
             f"{html_scan}"
             f"</div>")
    folium.CircleMarker(
        location=[r["lat"], r["lon"]], radius=9,
        color="#ffffff", weight=2,
        fill=True, fill_color=color_db(r["la_eq"]), fill_opacity=1.0,
        tooltip=f"<b>{r['id']}</b>: {r['la_eq']} dB - {r['calle']}",
        popup=folium.Popup(popup, max_width=320),
    ).add_to(fg_med)
fg_med.add_to(m)

LayerControl(collapsed=False).add_to(m)
print("Mapa construido.")


Mapa construido.


In [16]:
# ============================================================
# 4. Panel con distribución de clases y leyenda
# ============================================================

conteo = gdf_parches.groupby("liu_class").size().sort_values(ascending=False)
total = int(conteo.sum())
barras = "".join(
    f"""<div class='barra-fila'>
        <span class='chip' style='background:{CLASES[c]['color']}'></span>
        <span class='barra-nombre'>{c}</span>
        <span class='barra-track'><span class='barra-fill'
              style='width:{conteo[c]/total*100:.0f}%;background:{CLASES[c]['color']}'></span></span>
        <span class='barra-num'>{conteo[c]} ({conteo[c]/total*100:.0f}%)</span>
    </div>"""
    for c in conteo.index
)

leyenda_vias = "".join(
    f"<div class='fila'><span class='linea' style='height:{v['peso']}px'></span>{k}</div>"
    for k, v in vias.items()
)

panel = f"""
<style>
  .stg-panel {{
    position: fixed; top: 14px; left: 60px; z-index: 9999;
    background: rgba(255,255,255,0.97); border-radius: 14px;
    box-shadow: 0 4px 20px rgba(0,0,0,0.20);
    padding: 16px 20px; font-family: system-ui, sans-serif; width: 360px;
  }}
  .stg-cerrar {{
    position: absolute; top: 8px; right: 10px; border: none; background: none;
    font-size: 15px; color: #99a; cursor: pointer; padding: 2px 6px; border-radius: 6px;
  }}
  .stg-cerrar:hover {{ background: #eee; color: #333; }}
  .stg-chip {{
    display: none; position: fixed; z-index: 9998;
    border: none; border-radius: 999px; padding: 7px 14px;
    background: rgba(255,255,255,0.97); box-shadow: 0 3px 12px rgba(0,0,0,0.22);
    font-size: 12px; color: #223; cursor: pointer;
    font-family: system-ui, sans-serif;
  }}
  .stg-chip:hover {{ background: #f0f2f8; }}
  #chip-stg-panel {{ top: 14px; left: 60px; }}
  #chip-stg-leyenda {{ bottom: 24px; left: 12px; }}
  .stg-panel h3 {{ margin: 0; font-size: 16px; color: #14141f; }}
  .stg-panel .sub {{ font-size: 11.5px; color: #667; margin: 2px 0 10px; }}
  .barra-fila {{ display: flex; align-items: center; gap: 7px; margin: 3px 0; font-size: 11.5px; }}
  .chip {{ width: 12px; height: 12px; border-radius: 3px; flex: none; border: 1px solid rgba(0,0,0,.25); }}
  .barra-nombre {{ width: 108px; flex: none; color: #333; }}
  .barra-track {{ flex: 1; height: 7px; background: #eee; border-radius: 4px; overflow: hidden; }}
  .barra-fill {{ display: block; height: 100%; border-radius: 4px; }}
  .barra-num {{ width: 74px; flex: none; text-align: right; color: #555; font-variant-numeric: tabular-nums; }}
  .stg-hint {{ font-size: 11px; color: #778; margin-top: 9px; }}
  .stg-leyenda {{
    position: fixed; bottom: 24px; left: 12px; z-index: 9999;
    background: rgba(255,255,255,0.97); border-radius: 12px;
    box-shadow: 0 4px 18px rgba(0,0,0,0.18);
    padding: 12px 16px; font-family: system-ui, sans-serif; font-size: 11px;
  }}
  .stg-leyenda b {{ display: block; margin: 6px 0 4px; }}
  .stg-leyenda .fila {{ display: flex; align-items: center; gap: 7px; margin: 3px 0; }}
  .stg-leyenda .linea {{ width: 22px; background: #2d2d2d; border-radius: 2px; display: inline-block; }}
  .stg-leyenda .punto {{ width: 11px; height: 11px; border-radius: 50%; border: 2px solid #fff;
                          box-shadow: 0 0 0 1px #999; display: inline-block; }}
  .stg-leyenda .raya {{ width: 22px; border-top: 2px dashed #4a4a4a; display: inline-block; }}
</style>
<button class="stg-chip" id="chip-stg-panel" onclick="stgMostrar('stg-panel')">ℹ Descripción y resultados</button>
<button class="stg-chip" id="chip-stg-leyenda" onclick="stgMostrar('stg-leyenda')">🗺 Leyenda</button>
<div class="stg-panel" id="stg-panel">
  <button class="stg-cerrar" title="Ocultar panel" onclick="stgOcultar('stg-panel')">✕</button>
  <h3>{NOMBRE_CIUDAD.capitalize()} — Morfología de red vial</h3>
  <div class="sub">6 clases de Liu et al. 2025 · SGBNet (GNN) en GPU ·
  {total} parches clasificados · datos OpenStreetMap</div>
  {barras}
  <div class="stg-hint">Cursor sobre un parche: resumen. <b>Clic sobre un parche: detalle
  completo</b> (probabilidades del modelo). Los puntos son mediciones acústicas reales de
  enero 2026 — clic para el detalle.</div>
</div>
<div class="stg-leyenda" id="stg-leyenda">
  <button class="stg-cerrar" title="Ocultar leyenda" onclick="stgOcultar('stg-leyenda')">✕</button>
  <b>Red vial estructurante</b>{leyenda_vias}
  <b>Otros</b>
  <div class="fila"><span class="raya"></span>Límite comunal</div>
  <div class="fila"><span class="punto" style="background:#c62828"></span>Medición ≥ 70 dB</div>
  <div class="fila"><span class="punto" style="background:#ef6c00"></span>65–70 dB</div>
  <div class="fila"><span class="punto" style="background:#f9a825"></span>60–65 dB</div>
  <div class="fila"><span class="punto" style="background:#2e7d32"></span>&lt; 60 dB</div>
</div>
"""

toggle_js = """
<script>
function stgOcultar(id) {
  document.getElementById(id).style.display = 'none';
  document.getElementById('chip-' + id).style.display = 'inline-block';
}
function stgMostrar(id) {
  document.getElementById(id).style.display = 'block';
  document.getElementById('chip-' + id).style.display = 'none';
}
</script>
"""

m.get_root().html.add_child(folium.Element(panel + toggle_js))
print("Panel agregado.")


Panel agregado.


In [17]:
# ============================================================
# 5. Detalle completo del parche al hacer clic (JS, sin peso extra)
# ============================================================

clases_js = json.dumps(
    {v["clave"]: {"nombre": v["nombre"], "color": v["color"], "desc": v["descripcion_es"]}
     for v in CLASES.values()},
    ensure_ascii=False,
)

click_js = """
<script>
window.addEventListener("load", function () {
  var capa = window["__CAPA__"];
  var mapa = window["__MAPA__"];
  if (!capa || !mapa) { console.warn("capa/mapa no encontrados"); return; }
  var CL = __CLASES__;

  function fila(k, v) {
    return "<tr><td style='padding:1px 10px 1px 0;color:#556'>" + k +
           "</td><td style='font-weight:600'>" + v + "</td></tr>";
  }

  capa.on("click", function (e) {
    var p = e.layer.feature.properties;
    var info = null;
    for (var k in CL) if (CL[k].nombre === p.liu_class) info = CL[k];

    var html = "<div style='font-family:system-ui;font-size:12px;min-width:250px'>";
    html += "<h4 style='margin:0 0 2px'>Parche " + p.id + " — " + p.liu_class + "</h4>";
    if (info) html += "<div style='color:#667;margin-bottom:7px'>" + info.desc + "</div>";
    html += "<table>";
    html += fila("Confianza", p.confianza != null ? (p.confianza * 100).toFixed(1) + " %" : "s/d");
    html += fila("Bloques (manzanas)", p.n_bloques);
    html += fila("Tamaño", "1 × 1 km (Web Mercator)");
    html += "</table>";

    if (p.probs) {
      var probs = JSON.parse(p.probs);
      var orden = Object.keys(probs).sort(function (a, b) { return probs[b] - probs[a]; });
      html += "<div style='margin:9px 0 3px;font-weight:600'>Probabilidades del modelo</div>";
      orden.forEach(function (k) {
        var c = CL[k] || { nombre: k, color: "#999" };
        var pct = probs[k] * 100;
        html += "<div style='display:flex;align-items:center;gap:6px;margin:2px 0'>" +
          "<span style='width:112px;flex:none;color:#334'>" + c.nombre + "</span>" +
          "<span style='flex:1;height:8px;background:#eee;border-radius:4px;overflow:hidden'>" +
          "<span style='display:block;height:100%;width:" + Math.max(pct, 1).toFixed(0) +
          "%;background:" + c.color + "'></span></span>" +
          "<span style='width:46px;text-align:right;font-variant-numeric:tabular-nums'>" +
          pct.toFixed(1) + "%</span></div>";
      });
    } else {
      html += "<div style='margin-top:7px;color:#889'>Parche sin bloques suficientes: " +
              "Sparse por definición (no pasó por el modelo).</div>";
    }
    html += "<div style='margin-top:9px;font-size:10.5px;color:#899'>" +
            "SGBNet reentrenado (RTX 5060) — acc. 0.775 / F1 0.744 en test USNMS</div></div>";

    L.popup({ maxWidth: 340 }).setLatLng(e.latlng).setContent(html).openOn(mapa);
  });
});
</script>
"""

click_js = (click_js
            .replace("__CAPA__", gj_parches.get_name())
            .replace("__MAPA__", m.get_name())
            .replace("__CLASES__", clases_js))

m.get_root().html.add_child(folium.Element(click_js))
print("Popup de detalle por clic agregado (capa:", gj_parches.get_name() + ")")


Popup de detalle por clic agregado (capa: geo_json_f4cf5bbdc848004c6975b780cb07c916)


In [18]:
# ============================================================
# 6. Guardar
# ============================================================

ruta = APP / f"mapa_{NOMBRE_CIUDAD.lower()}.html"
m.save(str(ruta))
import os
print("App guardada:", ruta, f"({os.path.getsize(ruta)/1e6:.1f} MB)")


App guardada: 03_app/mapa_santiago.html (10.2 MB)
